# Phase 5 Comic Project Persistence Validation

This notebook exercises the persisted comic project workflow.

It can:

- create a new local comic project under `local_data/comics`
- append a new page to an existing comic project
- list stored comic projects

Before running it, start the 3.0 server:

```powershell
npm.cmd run dev:server
```


In [ ]:
from __future__ import annotations

import base64
import mimetypes
import json
from pathlib import Path

import requests
from IPython.display import Image as IPyImage, SVG, display

repo_root = Path(r"f:\Documents\GitHub\AI_TRPG_616\version 3.0")
api_base_url = "http://127.0.0.1:4316"

print("Repo root:", repo_root)
print("API base URL:", api_base_url)


In [ ]:
def file_to_data_url(path: str | Path) -> str:
    file_path = Path(path)
    if not file_path.exists():
        raise FileNotFoundError(f"Image file not found: {file_path}")

    mime_type = mimetypes.guess_type(file_path.name)[0] or "application/octet-stream"
    return f"data:{mime_type};base64,{base64.b64encode(file_path.read_bytes()).decode('ascii')}"


def ensure_server_available() -> None:
    try:
        response = requests.get(f"{api_base_url}/api/health", timeout=10)
        response.raise_for_status()
    except Exception as exc:
        raise RuntimeError(
            "AI TRPG 3.0 server is not reachable at http://127.0.0.1:4316. "
            "Start it in a separate terminal with: cd 'f:\\Documents\\GitHub\\AI_TRPG_616\\version 3.0' ; npm.cmd run dev:server"
        ) from exc


def get_json(path: str):
    response = requests.get(f"{api_base_url}{path}", timeout=120)
    if not response.ok:
        raise RuntimeError(f"{response.status_code} {response.text}")
    return response.json()


def post_json(path: str, payload: dict):
    response = requests.post(f"{api_base_url}{path}", json=payload, timeout=300)
    if not response.ok:
        raise RuntimeError(f"{response.status_code} {response.text}")
    return response.json()


def display_image_from_path(path: str | Path) -> None:
    target = Path(path)
    if target.suffix.lower() == ".svg":
        display(SVG(filename=str(target)))
    else:
        display(IPyImage(filename=str(target)))


In [ ]:
# Choose one mode: "list", "create", or "append".
MODE = "create"

# For append mode, fill this in with an existing comicId.
COMIC_ID = None

IMAGE_PROFILE_ID = "mock-image"
# IMAGE_PROFILE_ID = "gemini-image"

STYLE_ID = "american-modern"
TITLE = ""
DESCRIPTION = ""
GENERATE_METADATA = True
METADATA_MODEL_ACCESS_MODE = "mock"
METADATA_MODEL_PROFILE_ID = "chatgpt"
METADATA_RUNTIME_MODEL_CONFIG = {}
RUNTIME_IMAGE_MODEL_CONFIG = {}

STORY_PROMPT = """A rookie exorcist enters a flooded subway station under Tokyo at midnight and discovers a ghost train that only appears to people marked for death."""
NEGATIVE_PROMPT = "blurry unreadable text broken anatomy duplicated faces"
ALLOW_FALLBACK = True
STORY_MEMORY_SUMMARY = ""

REFERENCE_IMAGES = [
    # {
    #     "role": "character",
    #     "name": "Rin",
    #     "appearance": "young Japanese woman, short dark hair, school uniform under a raincoat, carrying prayer beads",
    #     "image_path": repo_root / "local_data" / "sample_refs" / "rin.png",
    # }
]


In [ ]:
ensure_server_available()
normalized_reference_images = []
for item in REFERENCE_IMAGES:
    normalized_item = {
        "role": item.get("role", "character")
    }
    if item.get("name"):
        normalized_item["name"] = str(item["name"])
    if item.get("appearance"):
        normalized_item["appearance"] = str(item["appearance"])
    if item.get("image_path"):
        normalized_item["imageUrl"] = file_to_data_url(item["image_path"])
    elif item.get("imageUrl"):
        normalized_item["imageUrl"] = str(item["imageUrl"])
    else:
        raise ValueError(f"Missing image path for reference image: {item}")
    normalized_reference_images.append(normalized_item)

if MODE == "list":
    result = get_json("/api/comics/projects")
    print(json.dumps(result, ensure_ascii=False, indent=2))

elif MODE == "create":
    payload = {
        "storyPrompt": STORY_PROMPT,
        "styleId": STYLE_ID,
        "referenceImages": normalized_reference_images,
        "negativePrompt": NEGATIVE_PROMPT,
        "allowFallback": ALLOW_FALLBACK,
        "imageProfileId": IMAGE_PROFILE_ID,
        "runtimeImageModelConfig": RUNTIME_IMAGE_MODEL_CONFIG,
        "title": TITLE,
        "description": DESCRIPTION,
        "generateMetadata": GENERATE_METADATA,
        "metadataLocale": "zh-CN",
        "metadataModelAccessMode": METADATA_MODEL_ACCESS_MODE,
        "metadataModelProfileId": METADATA_MODEL_PROFILE_ID,
        "metadataRuntimeModelConfig": METADATA_RUNTIME_MODEL_CONFIG,
    }
    result = post_json("/api/comics/projects", payload)
    print(json.dumps({
        "comicId": result["project"]["comicId"],
        "title": result["project"]["title"],
        "pageCount": result["project"]["pageCount"],
        "storageRoot": result["project"]["storageRoot"],
        "latestPageImage": result["page"]["image"]["storagePath"],
    }, ensure_ascii=False, indent=2))
    display_image_from_path(result["page"]["image"]["storagePath"])

elif MODE == "append":
    if not COMIC_ID:
        raise ValueError("COMIC_ID is required in append mode.")

    payload = {
        "storyPrompt": STORY_PROMPT,
        "referenceImages": normalized_reference_images,
        "negativePrompt": NEGATIVE_PROMPT,
        "storyMemorySummary": STORY_MEMORY_SUMMARY,
        "allowFallback": ALLOW_FALLBACK,
        "imageProfileId": IMAGE_PROFILE_ID,
        "runtimeImageModelConfig": RUNTIME_IMAGE_MODEL_CONFIG,
    }
    result = post_json(f"/api/comics/projects/{COMIC_ID}/pages", payload)
    print(json.dumps({
        "comicId": result["project"]["comicId"],
        "title": result["project"]["title"],
        "pageCount": result["project"]["pageCount"],
        "latestPageNumber": result["page"]["pageNumber"],
        "latestPageImage": result["page"]["image"]["storagePath"],
    }, ensure_ascii=False, indent=2))
    display_image_from_path(result["page"]["image"]["storagePath"])

else:
    raise ValueError(f"Unknown MODE: {MODE}")


In [ ]:
# Optional: inspect one stored project after create/append.
if MODE in {"create", "append"}:
    comic_id = result["project"]["comicId"]
    stored = get_json(f"/api/comics/projects/{comic_id}")
    print(json.dumps({
        "comicId": stored["comicId"],
        "title": stored["title"],
        "pageCount": stored["pageCount"],
        "references": len(stored["references"]),
        "pages": [page["pageNumber"] for page in stored["pages"]],
    }, ensure_ascii=False, indent=2))
